In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kgan31/doha-simple-context/dohas_nlp_ready_simple_lang.csv


In [3]:
# Install required libraries
!pip install sentence-transformers -q

import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F
from IPython.display import display

# ── Device Setup ─────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ── Load LaBSE directly on GPU ───────────────────────────────────
print("\nLoading LaBSE model...")
labse = SentenceTransformer("sentence-transformers/LaBSE", device=device)
print(f"LaBSE loaded on {device} ✅")

# ── Load Your Dataset ────────────────────────────────────────────
df = pd.read_csv("/kaggle/input/datasets/kgan31/doha-simple-context/dohas_nlp_ready_simple_lang.csv")
print(f"\nDataset shape: {df.shape}")
display(df.head(3))

# ── Clean Text ───────────────────────────────────────────────────
def clean_doha(text):
    if pd.isna(text):
        return ""
    return (
        str(text)
        .replace("\n", " ")
        .replace("।", "")
        .replace("॥", "")
        .strip()
    )

def clean_meaning(text):
    if pd.isna(text):
        return ""
    return str(text).strip()

df["doha_clean"]    = df["Doha"].apply(clean_doha)
df["meaning_clean"] = df["Meaning"].apply(clean_meaning)

df = df[
    (df["doha_clean"].str.len() > 0) &
    (df["meaning_clean"].str.len() > 0)
].reset_index(drop=True)

print(f"\nClean dataset size: {len(df)} rows")

# ── Compute Embeddings on GPU ────────────────────────────────────
# Larger batch size since we're on GPU
BATCH_SIZE = 128

print("\nComputing embeddings for dohas...")
doha_embeddings = labse.encode(
    df["doha_clean"].tolist(),
    batch_size           = BATCH_SIZE,
    show_progress_bar    = True,
    convert_to_tensor    = True,       # keeps tensor on GPU
    normalize_embeddings = True,       # L2 normalize → cosine = dot product
    device               = device
)
print(f"Doha embeddings shape    : {doha_embeddings.shape} | device: {doha_embeddings.device}")

print("\nComputing embeddings for meanings...")
meaning_embeddings = labse.encode(
    df["meaning_clean"].tolist(),
    batch_size           = BATCH_SIZE,
    show_progress_bar    = True,
    convert_to_tensor    = True,
    normalize_embeddings = True,
    device               = device
)
print(f"Meaning embeddings shape : {meaning_embeddings.shape} | device: {meaning_embeddings.device}")

# ── Cosine Similarity on GPU ─────────────────────────────────────
# Both tensors already on GPU — entire operation stays on GPU
# No CPU transfer until we need numpy for pandas
print("\nComputing similarities on GPU...")
with torch.no_grad():
    # Since L2 normalized: cosine similarity = dot product
    similarities_tensor = (doha_embeddings * meaning_embeddings).sum(dim=-1)

    # Verify computation happened on GPU
    print(f"Similarity tensor device: {similarities_tensor.device}")

# Move to CPU only once for pandas/numpy operations
similarities = similarities_tensor.cpu().numpy()
df["similarity_score"] = similarities

# Clear GPU cache after embedding computation
if device.type == "cuda":
    torch.cuda.empty_cache()
    print(f"GPU memory freed. Current allocated: "
          f"{torch.cuda.memory_allocated(0)/1e6:.1f} MB")

# ── Results Summary ──────────────────────────────────────────────
print("\n" + "="*55)
print("SIMILARITY SCORE SUMMARY")
print("="*55)
print(f"Mean   similarity : {similarities.mean():.4f}")
print(f"Median similarity : {np.median(similarities):.4f}")
print(f"Min    similarity : {similarities.min():.4f}")
print(f"Max    similarity : {similarities.max():.4f}")
print(f"Std    deviation  : {similarities.std():.4f}")

print("\nScore Distribution:")
buckets = [
    ("Very High  (0.8–1.0)", (similarities >= 0.8).sum()),
    ("High       (0.6–0.8)", ((similarities >= 0.6) & (similarities < 0.8)).sum()),
    ("Medium     (0.4–0.6)", ((similarities >= 0.4) & (similarities < 0.6)).sum()),
    ("Low        (0.2–0.4)", ((similarities >= 0.2) & (similarities < 0.4)).sum()),
    ("Very Low   (0.0–0.2)", (similarities < 0.2).sum()),
]
for label, count in buckets:
    pct = count / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"  {label} : {count:4d} ({pct:5.1f}%) {bar}")

# ── Pairwise Similarity Matrix on GPU (optional) ─────────────────
# Useful to check if meanings are unique / dohas are not duplicates
# Only run if dataset is small enough (< 5000 rows)
if len(df) <= 5000:
    print("\nComputing pairwise doha similarity matrix on GPU...")
    with torch.no_grad():
        # Matrix multiply: (N, D) x (D, N) = (N, N)
        pairwise = torch.mm(doha_embeddings, doha_embeddings.T)
        
        # Zero out diagonal (self-similarity)
        pairwise.fill_diagonal_(0)
        
        # Find most similar pairs (potential duplicates)
        top_vals, top_idx = pairwise.topk(k=3, dim=1)
    
    top_vals = top_vals.cpu().numpy()
    top_idx  = top_idx.cpu().numpy()
    
    print("\nTop 5 Most Similar Doha Pairs (potential duplicates):")
    seen = set()
    count = 0
    for i in range(len(df)):
        j     = top_idx[i][0]
        score = top_vals[i][0]
        pair  = tuple(sorted([i, j]))
        if pair not in seen and score > 0.85:
            seen.add(pair)
            print(f"\n  Score: {score:.4f}")
            print(f"  Doha A: {df['doha_clean'].iloc[i][:70]}...")
            print(f"  Doha B: {df['doha_clean'].iloc[j][:70]}...")
            count += 1
            if count >= 5:
                break
    
    if count == 0:
        print("  No near-duplicate dohas found (score > 0.85) ✅")
    
    del pairwise
    if device.type == "cuda":
        torch.cuda.empty_cache()

# ── Top 10 Most Aligned ──────────────────────────────────────────
print("\n" + "="*55)
print("TOP 10 — Most Aligned (Doha ↔ Meaning)")
print("="*55)
top10 = df.nlargest(10, "similarity_score")[
    ["Author", "Theme", "doha_clean", "meaning_clean", "similarity_score"]
]
for rank, (i, row) in enumerate(top10.iterrows(), 1):
    print(f"\nRank {rank} | Score: {row['similarity_score']:.4f}")
    print(f"  Author  : {row['Author']}")
    print(f"  Theme   : {row['Theme']}")
    print(f"  Doha    : {row['doha_clean'][:80]}...")
    print(f"  Meaning : {row['meaning_clean'][:80]}...")

# ── Bottom 10 Least Aligned ──────────────────────────────────────
print("\n" + "="*55)
print("BOTTOM 10 — Least Aligned (potential data issues)")
print("="*55)
bot10 = df.nsmallest(10, "similarity_score")[
    ["Author", "Theme", "doha_clean", "meaning_clean", "similarity_score"]
]
for rank, (i, row) in enumerate(bot10.iterrows(), 1):
    print(f"\nRank {rank} | Score: {row['similarity_score']:.4f}")
    print(f"  Author  : {row['Author']}")
    print(f"  Theme   : {row['Theme']}")
    print(f"  Doha    : {row['doha_clean'][:80]}...")
    print(f"  Meaning : {row['meaning_clean'][:80]}...")

# ── Per Theme Analysis ───────────────────────────────────────────
print("\n" + "="*55)
print("SIMILARITY BY THEME")
print("="*55)
theme_stats = (
    df.groupby("Theme")["similarity_score"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
    .round(4)
)
theme_stats.columns = ["Mean Score", "Median Score", "Count"]
display(theme_stats)

# ── Save Results ─────────────────────────────────────────────────
output_path = "/kaggle/working/doha_similarity_scores.csv"
df[[
    "Author", "Theme", "Doha", "Meaning",
    "Context", "doha_clean", "meaning_clean",
    "similarity_score"
]].to_csv(output_path, index=False)
print(f"\nResults saved to: {output_path} ✅")

# ── Threshold Recommendation ─────────────────────────────────────
print("\n" + "="*55)
print("THRESHOLD RECOMMENDATION FOR AUXILIARY LOSS")
print("="*55)
mean_score  = similarities.mean()
std_score   = similarities.std()
recommended = round(float(mean_score - std_score), 2)
print(f"  Mean similarity                  : {mean_score:.4f}")
print(f"  Recommended min_semantic for loss: {recommended}")
print(f"  Recommended max_semantic for loss: 0.92")
print(f"\n  Use in PostProcessor:")
print(f"  PostProcessor(tokenizer,")
print(f"                min_semantic={recommended},")
print(f"                max_semantic=0.92)")

# ── Final GPU Memory Report ──────────────────────────────────────
if device.type == "cuda":
    print(f"\nFinal GPU memory allocated : "
          f"{torch.cuda.memory_allocated(0)/1e6:.1f} MB")
    print(f"Final GPU memory reserved  : "
          f"{torch.cuda.memory_reserved(0)/1e6:.1f} MB")

Using device: cuda
GPU: Tesla T4
Memory Available: 15.64 GB

Loading LaBSE model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LaBSE loaded on cuda ✅

Dataset shape: (7889, 6)


,Author,Doha,Theme,Meaning,URL,Context
0,रसलीन,मोर पच्छ जो सिर चढ़ै बारन तें अधिकाय।\r\nसहस च...,शृंगार,नायिका के केशों में लगे मोरपंखों की अनुपम शोभा...,https://kavitakosh.org/kk/%E0%A4%85%E0%A4%82%E...,मोरपंखी बाल
1,रसलीन,बेनी बधि इक ठौर ह्वै अहि सम राखत ठौर।\r\nबिथुर...,शृंगार,नायिका की बँधी हुई चोटी सर्पिणी के समान मनमोहक...,https://kavitakosh.org/kk/%E0%A4%85%E0%A4%82%E...,नागिन सी चोटी
2,रसलीन,जे हरि रह त्रिलोक मों कालीनाथ कहाइ।\r\nते तुव ...,शृंगार,नायिका की चोटी की मोहक शक्ति इतनी तीव्र है कि ...,https://kavitakosh.org/kk/%E0%A4%85%E0%A4%82%E...,चोटी का जादू


...

Doha embeddings shape    : torch.Size([7889, 768]) | device: cuda:0

Computing embeddings for meanings...


Batches:   0%|          | 0/62 [00:00<?, ?it/s]

Meaning embeddings shape : torch.Size([7889, 768]) | device: cuda:0

Computing similarities on GPU...
Similarity tensor device: cuda:0
GPU memory freed. Current allocated: 3831.4 MB

SIMILARITY SCORE SUMMARY
Mean   similarity : 0.4960
Median similarity : 0.5019
Min    similarity : -0.1146
Max    similarity : 0.9736
Std    deviation  : 0.1682

Score Distribution:
  Very High  (0.8–1.0) :  208 (  2.6%) █
  High       (0.6–0.8) : 2065 ( 26.2%) █████████████
  Medium     (0.4–0.6) : 3369 ( 42.7%) █████████████████████
  Low        (0.2–0.4) : 1887 ( 23.9%) ███████████
  Very Low   (0.0–0.2) :  360 (  4.6%) ██

TOP 10 — Most Aligned (Doha ↔ Meaning)

Rank 1 | Score: 0.9736
  Author  : चंद्रसिंह बिरकाली
  Theme   : रौद्र
  Doha    : जैसे-जैसे संसार के प्राणी सूखते जा रहे है उसी मात्रा में लूएं तेज होती जा रही है...
  Meaning : जैसे-जैसे संसार के प्राणी सूखते जा रहे हैं, वैसे-वैसे लूएँ और तेज़ होती जा रही ह...

Rank 2 | Score: 0.9410
  Author  : रामेश्वर काम्बोज ‘हिमांशु’
  Theme   : शत्रुता


,Mean Score,Median Score,Count
Theme,,,
दुःख,0.7454,0.7454,1
दिवस,0.6960,0.6978,8
योग,0.6889,0.7462,6
कला,0.6831,0.6831,1
उत्सव,0.6726,0.6726,1
...,...,...,...
लाज,0.3195,0.2981,4
आनंद,0.2610,0.2610,1
रूप,0.2593,0.2575,34



Results saved to: /kaggle/working/doha_similarity_scores.csv ✅

THRESHOLD RECOMMENDATION FOR AUXILIARY LOSS
  Mean similarity                  : 0.4960
  Recommended min_semantic for loss: 0.33
  Recommended max_semantic for loss: 0.92

  Use in PostProcessor:
  PostProcessor(tokenizer,
                min_semantic=0.33,
                max_semantic=0.92)

Final GPU memory allocated : 3831.4 MB
Final GPU memory reserved  : 3997.2 MB


In [5]:
# ── Install ───────────────────────────────────────────────────────
!pip install transformers sentencepiece -q

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset
from IPython.display import display

# ── Device Setup ─────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ── Load MuRIL ───────────────────────────────────────────────────
print("\nLoading MuRIL...")
MODEL_NAME      = "google/muril-base-cased"
muril_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
muril_model     = AutoModel.from_pretrained(MODEL_NAME).to(device)
muril_model.eval()
print("MuRIL loaded ✅")

# ── Mean Pooling ─────────────────────────────────────────────────
def mean_pool(token_embeddings, attention_mask):
    """
    Average token embeddings weighted by attention mask
    Gives one fixed-size vector per input sentence
    """
    mask_expanded = attention_mask.unsqueeze(-1).float()
    return (
        torch.sum(token_embeddings * mask_expanded, dim=1) /
        torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
    )

# ── Text Dataset for Batched Encoding ────────────────────────────
class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx]

# ── Encode Texts in Batches on GPU ───────────────────────────────
def encode_texts(texts, batch_size=64, max_length=128):
    """
    Encode list of strings into L2-normalized embeddings using MuRIL
    Returns tensor of shape (N, hidden_dim) on CPU
    """
    dataset    = TextDataset(texts)
    loader     = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_embeddings = []

    with torch.no_grad():
        for batch_texts in loader:
            encoded = muril_tokenizer(
                list(batch_texts),
                padding        = True,
                truncation     = True,
                max_length     = max_length,
                return_tensors = "pt"
            ).to(device)

            outputs    = muril_model(**encoded)

            # Mean pool over token dimension → (batch, hidden_dim)
            embeddings = mean_pool(
                outputs.last_hidden_state,
                encoded["attention_mask"]
            )

            # L2 normalize → cosine similarity = dot product
            embeddings = F.normalize(embeddings, p=2, dim=-1)

            all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)   # (N, hidden_dim)

# ── Load Dataset ─────────────────────────────────────────────────
df = pd.read_csv("/kaggle/input/datasets/kgan31/doha-simple-context/dohas_nlp_ready_simple_lang.csv")
print(f"\nDataset shape : {df.shape}")
display(df.head(3))

# ── Clean Text ───────────────────────────────────────────────────
def clean_doha(text):
    if pd.isna(text): return ""
    return (
        str(text)
        .replace("\n", " ")
        .replace("।", "")
        .replace("॥", "")
        .strip()
    )

def clean_text(text):
    if pd.isna(text): return ""
    return str(text).strip()

df["doha_clean"]    = df["Doha"].apply(clean_doha)
df["meaning_clean"] = df["Meaning"].apply(clean_text)
df["context_clean"] = df["Context"].apply(clean_text)

df = df[
    (df["doha_clean"].str.len()    > 0) &
    (df["meaning_clean"].str.len() > 0) &
    (df["context_clean"].str.len() > 0)
].reset_index(drop=True)

print(f"Clean dataset size: {len(df)} rows")

# ── Encode All Three Fields ───────────────────────────────────────
print("\nEncoding dohas with MuRIL...")
doha_emb    = encode_texts(df["doha_clean"].tolist())
print(f"Doha embeddings    : {doha_emb.shape}")

print("\nEncoding meanings with MuRIL...")
meaning_emb = encode_texts(df["meaning_clean"].tolist())
print(f"Meaning embeddings : {meaning_emb.shape}")

print("\nEncoding contexts with MuRIL...")
context_emb = encode_texts(df["context_clean"].tolist())
print(f"Context embeddings : {context_emb.shape}")

# ── Compute Similarities ──────────────────────────────────────────
# All on CPU now — embeddings already moved off GPU
doha_meaning_sim = (doha_emb * meaning_emb).sum(dim=-1).numpy()
doha_context_sim = (doha_emb * context_emb).sum(dim=-1).numpy()

df["muril_doha_meaning_sim"] = doha_meaning_sim
df["muril_doha_context_sim"] = doha_context_sim

# ── Summary Function ──────────────────────────────────────────────
def print_summary(scores, label):
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  Mean   : {scores.mean():.4f}")
    print(f"  Median : {np.median(scores):.4f}")
    print(f"  Std    : {scores.std():.4f}")
    print(f"  Min    : {scores.min():.4f}")
    print(f"  Max    : {scores.max():.4f}")

    print(f"\n  Score Distribution:")
    buckets = [
        ("Very High  (0.8–1.0)", (scores >= 0.8).sum()),
        ("High       (0.6–0.8)", ((scores >= 0.6) & (scores < 0.8)).sum()),
        ("Medium     (0.4–0.6)", ((scores >= 0.4) & (scores < 0.6)).sum()),
        ("Low        (0.2–0.4)", ((scores >= 0.2) & (scores < 0.4)).sum()),
        ("Very Low   (0.0–0.2)", (scores < 0.2).sum()),
    ]
    for bucket_label, count in buckets:
        pct = count / len(scores) * 100
        bar = "█" * int(pct / 2)
        print(f"  {bucket_label} : {count:4d} ({pct:5.1f}%) {bar}")

print_summary(doha_meaning_sim, "MuRIL — Doha ↔ Meaning Similarity")
print_summary(doha_context_sim, "MuRIL — Doha ↔ Context Similarity")

# ── Direct Comparison: MuRIL vs LaBSE ────────────────────────────
# Paste your LaBSE mean from previous run here
LABSE_MEANING_MEAN = 0.4960
LABSE_CONTEXT_MEAN = None    # fill in if you ran context test before

print(f"\n{'='*55}")
print("  MODEL COMPARISON")
print(f"{'='*55}")
print(f"  {'Metric':<35} {'LaBSE':>8}  {'MuRIL':>8}  {'Winner':>8}")
print(f"  {'-'*60}")
print(f"  {'Doha ↔ Meaning (mean)':<35} "
      f"{LABSE_MEANING_MEAN:>8.4f}  "
      f"{doha_meaning_sim.mean():>8.4f}  "
      f"{'MuRIL ✅' if doha_meaning_sim.mean() > LABSE_MEANING_MEAN else 'LaBSE ✅':>8}")
if LABSE_CONTEXT_MEAN:
    print(f"  {'Doha ↔ Context (mean)':<35} "
          f"{LABSE_CONTEXT_MEAN:>8.4f}  "
          f"{doha_context_sim.mean():>8.4f}  "
          f"{'MuRIL ✅' if doha_context_sim.mean() > LABSE_CONTEXT_MEAN else 'LaBSE ✅':>8}")

# ── Per Theme Breakdown ───────────────────────────────────────────
print(f"\n{'='*55}")
print("  MuRIL SIMILARITY BY THEME")
print(f"{'='*55}")
theme_stats = (
    df.groupby("Theme")[["muril_doha_meaning_sim", "muril_doha_context_sim"]]
    .agg(["mean", "count"])
    .round(4)
)
display(theme_stats)

# ── Top 5 / Bottom 5 on Meaning Similarity ───────────────────────
print(f"\n{'='*55}")
print("  TOP 5 — Most Aligned (Doha ↔ Meaning)")
print(f"{'='*55}")
for rank, (_, row) in enumerate(
    df.nlargest(5, "muril_doha_meaning_sim").iterrows(), 1
):
    print(f"\n  Rank {rank} | Score: {row['muril_doha_meaning_sim']:.4f}")
    print(f"  Theme   : {row['Theme']}")
    print(f"  Doha    : {row['doha_clean'][:80]}")
    print(f"  Meaning : {row['meaning_clean'][:80]}")

print(f"\n{'='*55}")
print("  BOTTOM 5 — Least Aligned (Doha ↔ Meaning)")
print(f"{'='*55}")
for rank, (_, row) in enumerate(
    df.nsmallest(5, "muril_doha_meaning_sim").iterrows(), 1
):
    print(f"\n  Rank {rank} | Score: {row['muril_doha_meaning_sim']:.4f}")
    print(f"  Theme   : {row['Theme']}")
    print(f"  Doha    : {row['doha_clean'][:80]}")
    print(f"  Meaning : {row['meaning_clean'][:80]}")

# ── Threshold Recommendation ──────────────────────────────────────
print(f"\n{'='*55}")
print("  THRESHOLD RECOMMENDATION")
print(f"{'='*55}")

for scores, label, field in [
    (doha_meaning_sim, "Meaning", "meaning"),
    (doha_context_sim, "Context", "context"),
]:
    mean_s  = scores.mean()
    std_s   = scores.std()
    min_thr = round(float(mean_s - std_s), 2)
    print(f"\n  Based on Doha ↔ {label}:")
    print(f"    Mean score       : {mean_s:.4f}")
    print(f"    Recommended min  : {min_thr}  (mean - 1 std)")
    print(f"    Recommended max  : 0.92")
    print(f"    Use in loss      : lambda_meaning * max(0, {min_thr} - sim)")

# ── Decision Guide ────────────────────────────────────────────────
print(f"\n{'='*55}")
print("  DECISION GUIDE")
print(f"{'='*55}")
best_mean = max(doha_meaning_sim.mean(), doha_context_sim.mean())
best_src  = "Meaning" if doha_meaning_sim.mean() > doha_context_sim.mean() \
            else "Context"

if best_mean >= 0.6:
    print(f"  ✅ Use MuRIL with {best_src} as auxiliary loss signal")
    print(f"     Mean {best_mean:.4f} ≥ 0.6 → reliable gradient signal")
elif best_mean >= 0.45:
    print(f"  ⚠️  Use Normalized auxiliary loss (compare vs gold doha baseline)")
    print(f"     Mean {best_mean:.4f} is moderate — raw similarity too noisy")
    print(f"     Use: loss = max(0, sim_gold - sim_generated)")
else:
    print(f"  ❌ Skip auxiliary loss during training")
    print(f"     Mean {best_mean:.4f} < 0.45 → too noisy to be useful")
    print(f"     Use similarity only in postprocessing filter instead")

# ── Save ─────────────────────────────────────────────────────────
out_path = "/kaggle/working/muril_similarity_scores.csv"
df[[
    "Author", "Theme", "Doha", "Meaning", "Context",
    "muril_doha_meaning_sim", "muril_doha_context_sim"
]].to_csv(out_path, index=False)
print(f"\nSaved to: {out_path} ✅")

if device.type == "cuda":
    torch.cuda.empty_cache()
    print(f"GPU memory cleared ✅")

Device : cuda
GPU    : Tesla T4
Memory : 15.64 GB

Loading MuRIL...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MuRIL loaded ✅

Dataset shape : (7889, 6)


,Author,Doha,Theme,Meaning,URL,Context
0,रसलीन,मोर पच्छ जो सिर चढ़ै बारन तें अधिकाय।\r\nसहस च...,शृंगार,नायिका के केशों में लगे मोरपंखों की अनुपम शोभा...,https://kavitakosh.org/kk/%E0%A4%85%E0%A4%82%E...,मोरपंखी बाल
1,रसलीन,बेनी बधि इक ठौर ह्वै अहि सम राखत ठौर।\r\nबिथुर...,शृंगार,नायिका की बँधी हुई चोटी सर्पिणी के समान मनमोहक...,https://kavitakosh.org/kk/%E0%A4%85%E0%A4%82%E...,नागिन सी चोटी
2,रसलीन,जे हरि रह त्रिलोक मों कालीनाथ कहाइ।\r\nते तुव ...,शृंगार,नायिका की चोटी की मोहक शक्ति इतनी तीव्र है कि ...,https://kavitakosh.org/kk/%E0%A4%85%E0%A4%82%E...,चोटी का जादू


Clean dataset size: 7889 rows

Encoding dohas with MuRIL...
Doha embeddings    : torch.Size([7889, 768])

Encoding meanings with MuRIL...
Meaning embeddings : torch.Size([7889, 768])

Encoding contexts with MuRIL...
Context embeddings : torch.Size([7889, 768])

  MuRIL — Doha ↔ Meaning Similarity
  Mean   : 0.9941
  Median : 0.9941
  Std    : 0.0020
  Min    : 0.9798
  Max    : 0.9995

  Score Distribution:
  Very High  (0.8–1.0) : 7889 (100.0%) ██████████████████████████████████████████████████
  High       (0.6–0.8) :    0 (  0.0%) 
  Medium     (0.4–0.6) :    0 (  0.0%) 
  Low        (0.2–0.4) :    0 (  0.0%) 
  Very Low   (0.0–0.2) :    0 (  0.0%) 

  MuRIL — Doha ↔ Context Similarity
  Mean   : 0.9913
  Median : 0.9915
  Std    : 0.0021
  Min    : 0.9774
  Max    : 0.9963

  Score Distribution:
  Very High  (0.8–1.0) : 7889 (100.0%) ██████████████████████████████████████████████████
  High       (0.6–0.8) :    0 (  0.0%) 
  Medium     (0.4–0.6) :    0 (  0.0%) 
  Low        (0.2–0

muril_doha_meaning_sim       muril_doha_context_sim      
                             mean count                   mean count
Theme                                                               
अज्ञान                     0.9927     1                 0.9909     1
अद्भुत                     0.9937    18                 0.9905    18
अध्यात्म                   0.9963     1                 0.9938     1
अन्य                       0.9858     1                 0.9881     1
अहंकार                     0.9935     3                 0.9894     3
...                           ...   ...                    ...   ...
सौंदर्य                    0.9928   320                 0.9894   320
स्मृति                     0.9964     4                 0.9910     4
स्वतंत्रता                 0.9936     2                 0.9921     2
स्वभाव                     0.9933    61                 0.9906    61
हास्य                      0.9943    16                 0.9914    16

[104 rows x 4 columns]


  TOP 5 — Most Aligned (Doha ↔ Meaning)

  Rank 1 | Score: 0.9995
  Theme   : शत्रुता
 दोष भला देते किसे, हम थे ज़िम्मेदार पीठ पर वार 
  Meaning : अपनों ने हम पर किए, सदा पीठ पर वार। दोष भला देते किसे, हम थे ज़िम्मेदार।

  Rank 2 | Score: 0.9991
  Theme   : प्रकृति
  Doha    : बालकों की टोलियाँ मौहल्ले में खड़ी नाच रही हैं  बादली उनको झुक-झुक कर देख रही है
  Meaning : बच्चों की टोलियाँ मोहल्ले में खड़ी नाच रही हैं। बादली उन्हें झुक-झुक कर देख रही 

  Rank 3 | Score: 0.9991
  Theme   : करुण
  Doha    : पत्तों की मरमर से चमक कर कोसों तक चौकड़ी भरने वाले हरिण प्यास के मारे सुध-बुध खो
  Meaning : पत्तों की सरसराहट से चमक कर कोसों तक चौकड़ी भरने वाले हरिण प्यास के मारे अपनी सु

  Rank 4 | Score: 0.9990
  Theme   : चंद्र
  Doha    : रात में चन्द्रमा बादलियों में रमण करता हुआ दिखाई देता है-एक क्षण तो भूरे पर्वतों
  Meaning : रात में चंद्रमा बादलों के बीच घूमता हुआ दिखाई देता है, एक क्षण भूरे पर्वतों पर, 

  Rank 5 | Score: 0.9988
  Theme   : वर्षा
  Doha    : उतर दिशा को झोंका आया और घनघोर व